In [ ]:
import os
import pandas as pd
file_path = r'C:\project\political_ner\Research notes(1-1608) (3).xlsx'
research_notes_df = pd.read_excel(file_path)

In [ ]:
# research_notes_df = research_notes_df.rename(columns={
#     'What are the most prominent people or parties you have seen on TikTok today?': 'accountseen_tiktok',
#     'Have you seen any political actors belonging to some political groups in the European Parliament on TikTok today?\n': 'visibility_tiktok',
#     'What are the most prominent people or parties you have seen on Instagram today?  ': 'accountseen_instagram',
#     'Have you seen any politicians from some political groups in the European Parliament on Instagram today?\n': 'visibility_instagram',
# })

research_notes_df = research_notes_df.rename(columns={
    'Please list the names of official or affiliated accounts where you have seen these people or parties:\n': 'accountnames_tiktok',
    'Please list the names of official or affiliated accounts where you have seen these people or parties:': 'accountnames_instagram',
})


research_notes_df.columns


In [ ]:
research_notes_df['accountnames_tiktok']

In [ ]:
import pandas as pd
import pycountry
import re

file_path = r'C:\project\political_ner\Research notes(1-1608) (3).xlsx'
research_notes_df = pd.read_excel(file_path)


research_notes_df = research_notes_df.rename(columns={
    'Please list the names of official or affiliated accounts where you have seen these people or parties:\n': 'accountnames_tiktok',
    'Please list the names of official or affiliated accounts where you have seen these people or parties:': 'accountnames_instagram',
})


replacement_dict = {
    ' ': '',
    'DE§': 'DE3',
    'BG4': 'BG3',
    'F13': 'FI3',
    'HRO': 'HR2',
    'HRS': 'HR2',
    'HU0': 'HU3',
}
col_name = 'Your identifier (e.g. PT1, FR3, BG2)\n'

research_notes_df[col_name] = research_notes_df[col_name].replace(replacement_dict)

research_notes_df['country code'] = research_notes_df[col_name].str[:2]

def get_country_name(code):
    try:
        country = pycountry.countries.get(alpha_2=code.upper())
        if country:
            return country.name
    except:
        pass
    return 'Unknown'

research_notes_df['country'] = research_notes_df['country code'].apply(get_country_name)

research_notes_df['id_variable'] = research_notes_df['ID'].astype(str) + '_' + research_notes_df['country']

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "", name)

column_mapping = {}
for col in research_notes_df.columns:
    simplified_col = col.strip().replace('\n', '').replace('\xa0', '').lower()
    column_mapping[simplified_col] = col

simplified_columns_to_merge = [
    'accountnames_tiktok',
    'accountnames_instagram'
]

columns_to_merge = [column_mapping.get(col, None) for col in simplified_columns_to_merge]

missing_columns = [simplified_columns_to_merge[i] for i, col in enumerate(columns_to_merge) if col is None]
if missing_columns:
    print("Could not find the following columns in the DataFrame:")
    print(missing_columns)
else:
    print("All columns matched successfully.")

if not missing_columns:
    countries = research_notes_df['country'].unique()

    for country in countries:
        df_country = research_notes_df[research_notes_df['country'] == country]
        safe_country_name = sanitize_filename(country)
        var_name = 'df_' + safe_country_name.replace(' ', '_')

        melted_df = pd.melt(
            df_country,
            id_vars=['id_variable'],
            value_vars=columns_to_merge,
            var_name='variable',
            value_name='merged'
        )

        melted_df['id_variable_column'] = melted_df['id_variable'] + '_' + melted_df['variable']

        result_df = melted_df[['id_variable_column', 'merged']]

        result_df = result_df.dropna(subset=['merged'])


        result_df['merged'] = result_df['merged'].str.replace(r'[;,]\s*', ' , ', regex=True)

        merged_var_name = var_name + '_merged'
        globals()[merged_var_name] = result_df

        result_df.to_excel(f"C:\\project\\political_ner\\Insta_TikTok_AccountNames\\{safe_country_name}_accountnames.xlsx", index=False)




        print(f"Processed data for {safe_country_name}")

else:
    print("Please adjust the 'simplified_columns_to_merge' list to match your DataFrame columns.")


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

model_name = "xlm-roberta-large-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

In [ ]:
countries = [
    "Portugal", "Unknown", "Finland", "Hungary", "Spain", "Poland", 
    "Sweden", "Germany", "Croatia", "Bulgaria", "France"
]

input_dir = r'C:\project\political_ner\Insta_TikTok_AccountNames'
output_dir = r'C:\project\political_ner\Insta_TikTok_AccountNames\NER_Identify'

for country in countries:
    try:
        input_file = f"{input_dir}/{country}_accountnames.xlsx"
        output_file = f"{output_dir}/{country}_NER.xlsx"

        # Load the dataset for the country
        df_country = pd.read_excel(input_file)

        results = []
        for index, row in df_country.iterrows():
            id_variable_column = row.get('id_variable_column', None)
            original_text = row.get('merged', None)

            if pd.notnull(original_text) and str(original_text).strip():
                entities = nlp(str(original_text))

                for entity in entities:
                    entity_type = entity.get('entity_group', 'N/A')
                    if entity_type in ['ORG', 'PER']:
                        entity_text = entity['word']
                        # Append the result to the list
                        results.append({
                            'id_variable_column': id_variable_column,
                            'original': original_text,
                            'NER': entity_text
                        })
            else:
                pass  

        df_country_NER = pd.DataFrame(results)

        df_country_NER.to_excel(output_file, index=False)

        print(f"Processed and saved NER results for {country}")

    except Exception as e:
        print(f"Error processing {country}: {e}")
